# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
!git clone https://github.com/suha-2004/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 244 (delta 124), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (244/244), 3.07 MiB | 18.29 MiB/s, done.
Resolving deltas: 100% (124/124), done.


In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    paper	     scripts   submission
CLAUDE.md  docs		notebooks  README.md	     SETUP.md  work
data	   GUIDE.md	outputs    requirements.txt  skills


In [4]:
!ls data/raw

content_refresh_anonymized.csv


In [5]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

# Historical features available before the prediction period
feature_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate"
]

# Build the feature vector
X = df[feature_cols].copy()

# Convert features to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Fill missing values using the median
X = X.fillna(X.median())

print("Number of feature columns:", len(feature_cols))
print("Feature columns:")
print(feature_cols)

print("\nFeature vector shape:", X.shape)

print("\nMissing values after preprocessing:")
print(X.isna().sum())

Number of feature columns: 7
Feature columns:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'engagement_rate']

Feature vector shape: (30000, 7)

Missing values after preprocessing:
impressions_prev_30d      0
clicks_prev_30d           0
sessions_prev_30d         0
content_age_days          0
days_since_last_update    0
ctr                       0
engagement_rate           0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
### Feature notes

| Feature | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| `impressions_prev_30d` | Number of impressions during the previous 30 days. | Missing values are filled with the median. | Yes |
| `clicks_prev_30d` | Number of clicks during the previous 30 days. | Missing values are filled with the median. | Yes |
| `sessions_prev_30d` | Number of sessions during the previous 30 days. | Missing values are filled with the median. | Yes |
| `content_age_days` | Age of the content in days. | Missing values are filled with the median. | Yes |
| `days_since_last_update` | Number of days since the content was last updated. | Missing values are filled with the median. | Yes |
| `ctr` | Click-through rate associated with the content. | Missing values are filled with the median. | Yes |
| `engagement_rate` | Engagement rate associated with the content. | Missing values are filled with the median. | Yes |

These features represent historical content and search-performance signals intended to be available before the prediction/decision point. No target label is included in the feature vector.

In [6]:
feature_notes = pd.DataFrame({
    "Feature": feature_cols,
    "Missing_before_fill": df[feature_cols].isna().sum().values,
    "Missing_after_fill": X.isna().sum().values,
    "Available_before_prediction": ["Yes"] * len(feature_cols)
})

feature_notes

,Feature,Missing_before_fill,Missing_after_fill,Available_before_prediction
0,impressions_prev_30d,0,0,Yes
1,clicks_prev_30d,0,0,Yes
2,sessions_prev_30d,0,0,Yes
3,content_age_days,0,0,Yes
4,days_since_last_update,0,0,Yes
5,ctr,0,0,Yes
6,engagement_rate,0,0,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
### Leakage check

I checked the selected features for direct target leakage and outcome-period information.

The target-related outcome fields are kept separate from the feature vector. The feature vector contains only the selected predictor fields.

I also checked for fields containing `last_30d` that could represent the outcome period. Any such fields are excluded from the feature vector.

In [7]:
# Check the selected feature columns for possible leakage

# Columns that represent the recent outcome period
outcome_period_cols = [
    col for col in df.columns
    if "last_30d" in col.lower()
]

# Check whether any outcome-period columns are being used as features
outcome_features_used = set(feature_cols).intersection(outcome_period_cols)

print("Outcome-period columns found in dataset:")
print(outcome_period_cols)

print("\nOutcome-period columns used as features:")
print(outcome_features_used)

# Check for obvious target-related fields
target_related_terms = [
    "declining",
    "target",
    "label"
]

target_like_features = [
    col for col in feature_cols
    if any(term in col.lower() for term in target_related_terms)
]

print("\nTarget-like columns used as features:")
print(target_like_features)

# Overall check
if len(outcome_features_used) == 0 and len(target_like_features) == 0:
    print("\nLeakage check: PASS")
else:
    print("\nLeakage check: REVIEW")

Outcome-period columns found in dataset:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

Outcome-period columns used as features:
set()

Target-like columns used as features:
[]

Leakage check: PASS


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
### Excluded fields

- `impressions_last_30d`: Excluded because it represents the recent outcome period and could introduce information from the period being evaluated.
- `clicks_last_30d`: Excluded because it represents the recent outcome period and could introduce information from the period being evaluated.
- `sessions_last_30d`: Excluded because it represents the recent outcome period and could introduce information from the period being evaluated.
- `content_id`: Excluded because it is an identifier rather than a predictive content-performance signal.
- `client_id`: Excluded because it identifies the client rather than representing a generalizable content-performance signal.
- `trend_direction`: Excluded because it summarizes the observed trend and could overlap with the outcome being modeled.
- `trend_pct`: Excluded because it summarizes recent change and could contain information related to the outcome period.
- `model_used`: Excluded because it describes the model/provider used to generate data rather than the content's search-performance behavior.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = pd.DataFrame({
    "Excluded field": [
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "content_id",
        "client_id",
        "trend_direction",
        "trend_pct",
        "model_used"
    ],
    "Reason": [
        "Outcome-period information",
        "Outcome-period information",
        "Outcome-period information",
        "Identifier, not a predictive signal",
        "Client identifier, not a generalizable predictive signal",
        "Summarizes observed trend and may overlap with the outcome",
        "Summarizes recent change and may overlap with the outcome",
        "Describes data/model provenance rather than content performance"
    ]
})

excluded_fields

,Excluded field,Reason
0,impressions_last_30d,Outcome-period information
1,clicks_last_30d,Outcome-period information
2,sessions_last_30d,Outcome-period information
3,content_id,"Identifier, not a predictive signal"
4,client_id,"Client identifier, not a generalizable predict..."
5,trend_direction,Summarizes observed trend and may overlap with...
6,trend_pct,Summarizes recent change and may overlap with ...
7,model_used,Describes data/model provenance rather than co...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.